# Ollama Benchmarking Lab (CPU/RAM/Latency)

This notebook helps students benchmark local inference:
- Latency per prompt
- CPU and RAM usage (Python `psutil`)
- Throughput estimates (rough)

Run Ollama locally or via Docker Compose.

In [ ]:
import os
import requests

# Resolve Ollama endpoint depending on environment
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Using Ollama at:', BASE_URL)

# Diagnostic check
try:
    r = requests.get(BASE_URL, timeout=3)
    print('Server reachable. Status:', r.status_code)
    try:
        tags = requests.get(f"{BASE_URL}/api/tags", timeout=3)
        print('Tags endpoint status:', tags.status_code)
    except Exception as e:
        print('Tags endpoint check failed:', e)
except Exception as e:
    print('Server not reachable:', e)


In [ ]:
import os, time, statistics
import requests
import psutil

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")

def ollama_generate(prompt: str, model: str = MODEL):
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False}
    r = requests.post(url, json=payload, timeout=180)
    r.raise_for_status()
    return r.json()["response"]

print("OLLAMA_BASE_URL =", OLLAMA_BASE_URL)
print("MODEL =", MODEL)


## 1) Define a benchmark set

In [ ]:
prompts = [
    "Explain overfitting in 3 bullet points.",
    "Write a Python function that computes prime numbers up to N.",
    "Summarize the idea of attention in transformers in 5 sentences."
]

## 2) Measure latency

In [ ]:
latencies = []

for p in prompts:
    start = time.time()
    _ = ollama_generate(p)
    lat = time.time() - start
    latencies.append(lat)
    print(f"Prompt: {p[:40]}...  latency={lat:.2f}s")

print("\nLatency stats:")
print("min:", min(latencies))
print("max:", max(latencies))
print("mean:", statistics.mean(latencies))

## 3) Observe system usage (snapshot)

In [ ]:
cpu_percent = psutil.cpu_percent(interval=1)
mem = psutil.virtual_memory()

print("CPU% (system):", cpu_percent)
print("RAM used (GB):", round(mem.used/1e9, 2))
print("RAM total (GB):", round(mem.total/1e9, 2))
print("RAM percent:", mem.percent)

## 4) Mini-experiment: temperature effect

In [ ]:
def ollama_generate_temp(prompt: str, temperature: float, model: str = MODEL):
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {"model": model, "prompt": prompt, "temperature": temperature, "stream": False}
    r = requests.post(url, json=payload, timeout=180)
    r.raise_for_status()
    return r.json()["response"]

prompt = "Give a creative but accurate definition of entropy."
for t in [0.2, 0.7, 1.2]:
    out = ollama_generate_temp(prompt, temperature=t)
    print(f"\n--- temperature={t} ---\n{out}")
